In [ ]:
import sys
import importlib
import pickle
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.simple_env as simple_env
importlib.reload(simple_env)

from src.rl.simple_env import SimpleRuleEnv

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"

wid = 998

with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

env = SimpleRuleEnv(window_state, max_steps=1)

# 1) keep 一个攻击规则
env.reset()
_, r_attack_keep, _, info1 = env.step(rule_idx=0, action=0)

# 2) disable 一个攻击规则
env.reset()
_, r_attack_disable, _, info2 = env.step(rule_idx=0, action=1)

# 3) disable 一个正常规则（后10条里随便取一条，这里取第10条以后第一条）
env.reset()
_, r_normal_disable, _, info3 = env.step(rule_idx=10, action=1)

print("r_attack_keep:", r_attack_keep, info1)
print("r_attack_disable:", r_attack_disable, info2)
print("r_normal_disable:", r_normal_disable, info3)

In [ ]:
import sys
import importlib
import pickle
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.simple_env as simple_env
import src.rl.ac_model as ac_model

importlib.reload(simple_env)
importlib.reload(ac_model)

from src.rl.simple_env import SimpleRuleEnv
from src.rl.ac_model import ActorCriticNet

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"

wid = 998
with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

env = SimpleRuleEnv(window_state, max_steps=window_state["num_rules"])
model = ActorCriticNet(state_dim=12, action_dim=2, hidden_dim=64)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

gamma = 0.99
episode_rewards = []

for episode in range(100):
    state = env.reset()
    trajectory = []
    total_reward = 0.0

    for step in range(window_state["num_rules"]):
        rule_idx = step

        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        logits, value = model(state_tensor)
        probs = torch.softmax(logits, dim=-1).squeeze(0)
        action = torch.multinomial(probs, num_samples=1).item()

        next_state, reward, done, info = env.step(rule_idx, action)

        trajectory.append((state, action, reward))
        total_reward += reward
        state = next_state

        if done:
            break

    returns = []
    G = 0.0
    for _, _, r in reversed(trajectory):
        G = r + gamma * G
        returns.insert(0, G)

    states = torch.tensor(np.array([x[0] for x in trajectory], dtype=np.float32))
    actions = torch.tensor(np.array([x[1] for x in trajectory], dtype=np.int64))
    returns = torch.tensor(np.array(returns, dtype=np.float32).reshape(-1, 1))

    logits, values = model(states)
    log_probs = F.log_softmax(logits, dim=-1)
    selected_log_probs = log_probs.gather(1, actions.unsqueeze(1))

    advantages = returns - values
    policy_loss = -(selected_log_probs * advantages.detach()).mean()
    value_loss = F.mse_loss(values, returns)
    loss = policy_loss + 0.5 * value_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    episode_rewards.append(total_reward)

    if (episode + 1) % 10 == 0:
        print(
            f"episode={episode+1}, "
            f"total_reward={total_reward:.4f}, "
            f"policy_loss={policy_loss.detach().item():.4f}, "
            f"value_loss={value_loss.detach().item():.4f}"
        )

print("训练完成")
print("最后10个 episode reward:", episode_rewards[-10:])

In [ ]:
import sys
import pickle
from pathlib import Path
import pandas as pd
import torch

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

from src.rl.simple_env import SimpleRuleEnv
from src.rl.action_utils import ACTION_NAMES

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"
window_pool_dir = stream_dir / "window_rule_pools"

wid = 998

with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

rule_pool_df = pd.read_csv(window_pool_dir / f"window_{wid}_mixed_rule_pool.csv")

env = SimpleRuleEnv(window_state, max_steps=window_state["num_rules"])

state = env.reset()
records = []
total_reward = 0.0

for step in range(window_state["num_rules"]):
    rule_idx = step

    state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    logits, value = model(state_tensor)
    action = torch.argmax(logits, dim=-1).item()

    next_state, reward, done, info = env.step(rule_idx, action)
    total_reward += reward

    records.append({
        "rule_idx": rule_idx,
        "target_label": int(rule_pool_df.iloc[rule_idx]["target_label"]),
        "formula": rule_pool_df.iloc[rule_idx]["formula"],
        "action_name": ACTION_NAMES[action],
        "reward": reward
    })

    state = next_state
    if done:
        break

eval_df = pd.DataFrame(records)

print(eval_df[["rule_idx", "target_label", "action_name", "reward"]])
print("\n动作统计:")
print(eval_df["action_name"].value_counts())
print("\n按 target_label 分组统计:")
print(pd.crosstab(eval_df["target_label"], eval_df["action_name"]))
print("\ngreedy total_reward:", total_reward)

In [ ]:
import pickle
import numpy as np
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"

wid = 998
with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

active_mask = window_state["active_mask"].copy().astype(float)
weights = window_state["weights"].copy().astype(float)
initial_weights = window_state["initial_weights"].copy().astype(float)
rule_scores = window_state["rule_scores"].copy().astype(float)
target_labels = window_state["target_labels"].copy().astype(int)

def calc_reward(active_mask, weights, initial_weights, rule_scores, target_labels,
                attack_coef=2.0, normal_coef=1.0, drift_coef=0.2):
    pos_w = np.maximum(weights, 0.0)

    attack_contrib = np.sum(
        active_mask * (target_labels == 1) * rule_scores * pos_w
    ) / len(active_mask)

    normal_contrib = np.sum(
        active_mask * (target_labels == 0) * rule_scores * pos_w
    ) / len(active_mask)

    drift_penalty = np.mean(np.abs(weights - initial_weights))

    return (
        attack_coef * attack_contrib
        - normal_coef * normal_contrib
        - drift_coef * drift_penalty
    )

# 选一条正常规则和一条攻击规则
normal_idx = 10
attack_idx = 0

for normal_coef in [1.0, 1.5, 2.0, 2.5, 3.0]:
    base_reward = calc_reward(
        active_mask, weights, initial_weights, rule_scores, target_labels,
        attack_coef=2.0, normal_coef=normal_coef, drift_coef=0.2
    )

    # disable normal
    am1 = active_mask.copy()
    am1[normal_idx] = 0
    r_disable_normal = calc_reward(
        am1, weights, initial_weights, rule_scores, target_labels,
        attack_coef=2.0, normal_coef=normal_coef, drift_coef=0.2
    ) - base_reward

    # disable attack
    am2 = active_mask.copy()
    am2[attack_idx] = 0
    r_disable_attack = calc_reward(
        am2, weights, initial_weights, rule_scores, target_labels,
        attack_coef=2.0, normal_coef=normal_coef, drift_coef=0.2
    ) - base_reward

    print(
        f"normal_coef={normal_coef:.1f} | "
        f"disable_normal_delta={r_disable_normal:.6f} | "
        f"disable_attack_delta={r_disable_attack:.6f}"
    )

In [ ]:
import sys
import importlib
import pickle
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.reward_utils as reward_utils
import src.rl.simple_env as simple_env

importlib.reload(reward_utils)
importlib.reload(simple_env)

from src.rl.simple_env import SimpleRuleEnv

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"

wid = 998
with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

# keep 一个攻击规则
env = SimpleRuleEnv(window_state, max_steps=1)
env.reset()
_, r_attack_keep, _, _ = env.step(rule_idx=0, action=0)

# disable 一个攻击规则
env = SimpleRuleEnv(window_state, max_steps=1)
env.reset()
_, r_attack_disable, _, _ = env.step(rule_idx=0, action=1)

# disable 一个正常规则
env = SimpleRuleEnv(window_state, max_steps=1)
env.reset()
_, r_normal_disable, _, _ = env.step(rule_idx=10, action=1)

print("r_attack_keep:", r_attack_keep)
print("r_attack_disable:", r_attack_disable)
print("r_normal_disable:", r_normal_disable)

In [ ]:
import sys
import importlib
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.obs_utils as obs_utils
importlib.reload(obs_utils)

from src.rl.obs_utils import build_state_vector

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"

wid = 998
with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

active_mask = window_state["active_mask"]
weights = window_state["weights"]
rule_scores = window_state["rule_scores"]
target_labels = window_state["target_labels"]

# 构造监督数据：攻击规则->keep(0), 正常规则->disable(1)
X_list = []
y_list = []

for rule_idx in range(window_state["num_rules"]):
    state_vec = build_state_vector(
        active_mask, weights, rule_scores, rule_idx, target_labels
    )
    X_list.append(state_vec)

    action_label = 0 if target_labels[rule_idx] == 1 else 1
    y_list.append(action_label)

X = torch.tensor(np.array(X_list, dtype=np.float32))
y = torch.tensor(np.array(y_list, dtype=np.int64))

model_sup = nn.Sequential(
    nn.Linear(12, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)

optimizer = torch.optim.Adam(model_sup.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(200):
    logits = model_sup(X)
    loss = criterion(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        pred = logits.argmax(dim=1)
        acc = (pred == y).float().mean().item()
        print(f"epoch={epoch+1}, loss={loss.item():.6f}, acc={acc:.4f}")

with torch.no_grad():
    pred = model_sup(X).argmax(dim=1)
    acc = (pred == y).float().mean().item()

print("final acc:", acc)
print("pred:", pred.tolist())
print("true:", y.tolist())

In [ ]:
import sys
import importlib
import pickle
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.ac_model as ac_model
import src.rl.obs_utils as obs_utils
importlib.reload(ac_model)
importlib.reload(obs_utils)

from src.rl.ac_model import ActorCriticNet
from src.rl.obs_utils import build_state_vector

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"
model_dir = root / "outputs" / "models"
model_dir.mkdir(parents=True, exist_ok=True)

wid = 998
with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

active_mask = window_state["active_mask"]
weights = window_state["weights"]
rule_scores = window_state["rule_scores"]
target_labels = window_state["target_labels"]

# 构造监督数据：攻击规则->keep(0), 正常规则->disable(1)
X_list, y_list = [], []
for rule_idx in range(window_state["num_rules"]):
    state_vec = build_state_vector(
        active_mask, weights, rule_scores, rule_idx, target_labels
    )
    X_list.append(state_vec)
    y_list.append(0 if target_labels[rule_idx] == 1 else 1)

X = torch.tensor(np.array(X_list, dtype=np.float32))
y = torch.tensor(np.array(y_list, dtype=np.int64))

model_warm = ActorCriticNet(state_dim=12, action_dim=2, hidden_dim=64)
optimizer = torch.optim.Adam(model_warm.parameters(), lr=1e-3)

for epoch in range(300):
    logits, values = model_warm(X)
    cls_loss = F.cross_entropy(logits, y)

    optimizer.zero_grad()
    cls_loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        pred = logits.argmax(dim=1)
        acc = (pred == y).float().mean().item()
        print(f"epoch={epoch+1}, cls_loss={cls_loss.item():.6f}, acc={acc:.4f}")

with torch.no_grad():
    logits, _ = model_warm(X)
    pred = logits.argmax(dim=1)
    acc = (pred == y).float().mean().item()

save_path = model_dir / "window_998_actor_warmstart.pth"
torch.save(model_warm.state_dict(), save_path)

print("final acc:", acc)
print("pred:", pred.tolist())
print("true:", y.tolist())
print("saved:", save_path)

In [ ]:
import sys
import importlib
import pickle
from pathlib import Path
import pandas as pd
import torch

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.ac_model as ac_model
import src.rl.simple_env as simple_env
import src.rl.action_utils as action_utils

importlib.reload(ac_model)
importlib.reload(simple_env)
importlib.reload(action_utils)

from src.rl.ac_model import ActorCriticNet
from src.rl.simple_env import SimpleRuleEnv
from src.rl.action_utils import ACTION_NAMES

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"
window_pool_dir = stream_dir / "window_rule_pools"
model_dir = root / "outputs" / "models"

wid = 998

with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

rule_pool_df = pd.read_csv(window_pool_dir / f"window_{wid}_mixed_rule_pool.csv")

model = ActorCriticNet(state_dim=12, action_dim=2, hidden_dim=64)
model.load_state_dict(torch.load(model_dir / "window_998_actor_warmstart.pth", map_location="cpu"))
model.eval()

env = SimpleRuleEnv(window_state, max_steps=window_state["num_rules"])

state = env.reset()
records = []
total_reward = 0.0

for step in range(window_state["num_rules"]):
    rule_idx = step

    state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    logits, value = model(state_tensor)
    action = torch.argmax(logits, dim=-1).item()

    next_state, reward, done, info = env.step(rule_idx, action)
    total_reward += reward

    records.append({
        "rule_idx": rule_idx,
        "target_label": int(rule_pool_df.iloc[rule_idx]["target_label"]),
        "formula": rule_pool_df.iloc[rule_idx]["formula"],
        "action_name": ACTION_NAMES[action],
        "reward": reward
    })

    state = next_state
    if done:
        break

eval_df = pd.DataFrame(records)

print(eval_df[["rule_idx", "target_label", "action_name", "reward"]])
print("\n动作统计:")
print(eval_df["action_name"].value_counts())
print("\n按 target_label 分组统计:")
print(pd.crosstab(eval_df["target_label"], eval_df["action_name"]))
print("\ngreedy total_reward:", total_reward)

In [ ]:
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
log_dir = root / "outputs" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

attack_keep_rate = (
    ((eval_df["target_label"] == 1) & (eval_df["action_name"] == "keep")).sum()
    / (eval_df["target_label"] == 1).sum()
)

normal_disable_rate = (
    ((eval_df["target_label"] == 0) & (eval_df["action_name"] == "disable")).sum()
    / (eval_df["target_label"] == 0).sum()
)

selection_accuracy = (
    ((eval_df["target_label"] == 1) & (eval_df["action_name"] == "keep")).sum()
    + ((eval_df["target_label"] == 0) & (eval_df["action_name"] == "disable")).sum()
) / len(eval_df)

result_df = pd.DataFrame([{
    "window_id": 998,
    "attack_keep_rate": attack_keep_rate,
    "normal_disable_rate": normal_disable_rate,
    "selection_accuracy": selection_accuracy,
    "greedy_total_reward": total_reward
}])

save_path = log_dir / "window_998_metrics.csv"
result_df.to_csv(save_path, index=False)

print("saved:", save_path)
print(result_df)

In [ ]:
import sys
import pickle
import random
from pathlib import Path
import pandas as pd

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

from src.rl.simple_env import SimpleRuleEnv

stream_dir = root / "data" / "stream" / "swat"
window_state_dir = stream_dir / "window_rule_states"
log_dir = root / "outputs" / "logs"
log_dir.mkdir(parents=True, exist_ok=True)

wid = 998
with open(window_state_dir / f"window_{wid}_rule_state.pkl", "rb") as f:
    window_state = pickle.load(f)

def run_fixed_policy(state_dict, fixed_action):
    env = SimpleRuleEnv(state_dict, max_steps=state_dict["num_rules"])
    state = env.reset()
    total_reward = 0.0
    records = []

    for step in range(state_dict["num_rules"]):
        rule_idx = step
        next_state, reward, done, info = env.step(rule_idx, fixed_action)
        total_reward += reward
        records.append({
            "rule_idx": rule_idx,
            "target_label": int(state_dict["target_labels"][rule_idx]),
            "action": fixed_action,
            "reward": reward
        })
        state = next_state
        if done:
            break

    return total_reward, pd.DataFrame(records)

def run_random_policy(state_dict, seed=42):
    random.seed(seed)
    env = SimpleRuleEnv(state_dict, max_steps=state_dict["num_rules"])
    state = env.reset()
    total_reward = 0.0
    records = []

    for step in range(state_dict["num_rules"]):
        rule_idx = step
        action = random.randint(0, 1)
        next_state, reward, done, info = env.step(rule_idx, action)
        total_reward += reward
        records.append({
            "rule_idx": rule_idx,
            "target_label": int(state_dict["target_labels"][rule_idx]),
            "action": action,
            "reward": reward
        })
        state = next_state
        if done:
            break

    return total_reward, pd.DataFrame(records)

all_keep_reward, _ = run_fixed_policy(window_state, fixed_action=0)
all_disable_reward, _ = run_fixed_policy(window_state, fixed_action=1)
random_reward, _ = run_random_policy(window_state, seed=42)

baseline_df = pd.DataFrame([
    {"method": "WarmStart_greedy", "total_reward": 0.4996855284899333, "selection_accuracy": 0.95},
    {"method": "All_Keep", "total_reward": all_keep_reward, "selection_accuracy": None},
    {"method": "All_Disable", "total_reward": all_disable_reward, "selection_accuracy": None},
    {"method": "Random", "total_reward": random_reward, "selection_accuracy": None},
])

save_path = log_dir / "window_998_baseline_comparison.csv"
baseline_df.to_csv(save_path, index=False)

print("all_keep_reward:", all_keep_reward)
print("all_disable_reward:", all_disable_reward)
print("random_reward:", random_reward)
print("saved:", save_path)
print(baseline_df)